# 신경망 실습 과제 - 완전 구현

XOR 문제로 4가지 실험 수행

## 사용법:
1. 셀을 순서대로 실행하세요
2. 각 실험의 결과를 확인하세요
3. 파라미터를 바꿔가며 실험해보세요

## 실험 목록:
1. **활성화 함수 비교**: Sigmoid vs ReLU vs Tanh
2. **층 깊이 실험**: 1층, 2층, 5층 비교
3. **학습률 조정**: 0.01, 0.1, 1.0 비교
4. **기울기 확인**: 각 층의 기울기 크기 분석

In [ ]:
# ============================================
# 1. 라이브러리 import 및 기본 설정
# ============================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Mac의 경우)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('default')
np.random.seed(42)

print("✅ 라이브러리 로드 완료!")

In [ ]:
# ============================================
# 2. 기본 함수 정의 - 활성화 함수들
# ============================================

# 활성화 함수들
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    return x * (1 - x)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - x ** 2

print("✅ 활성화 함수 정의 완료!")

In [ ]:
# ============================================
# 3. XOR 데이터셋 정의
# ============================================

# XOR 데이터
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

print("✅ XOR 데이터셋 정의 완료!")
print(f"\nXOR 데이터셋:")
print(f"입력 X:\n{X}")
print(f"출력 y:\n{y.T}")

In [ ]:
# ============================================
# 4. 신경망 클래스 정의
# ============================================

class NeuralNetwork:
    def __init__(self, layer_sizes, activation='sigmoid', learning_rate=0.5):
        """
        layer_sizes: [입력, 은닉1, 은닉2, ..., 출력]
        activation: 'sigmoid', 'relu', 'tanh'
        """
        self.layer_sizes = layer_sizes
        self.activation = activation
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []
        self.loss_history = []
        self.gradient_norms = []

        # 가중치 초기화
        for i in range(len(layer_sizes) - 1):
            if activation == 'relu':
                w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2/layer_sizes[i])
            else:
                w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.5
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)

    def activate(self, x):
        if self.activation == 'sigmoid':
            return sigmoid(x)
        elif self.activation == 'relu':
            return relu(x)
        elif self.activation == 'tanh':
            return tanh(x)

    def activate_derivative(self, x):
        if self.activation == 'sigmoid':
            return sigmoid_derivative(x)
        elif self.activation == 'relu':
            return relu_derivative(x)
        elif self.activation == 'tanh':
            return tanh_derivative(x)

print("✅ 신경망 클래스 정의 완료!")

In [ ]:
# ============================================
# 5. 순전파 메서드 추가
# ============================================

def forward(self, X):
    self.activations = [X]
    self.z_values = []

    for i in range(len(self.weights)):
        z = self.activations[-1] @ self.weights[i] + self.biases[i]
        self.z_values.append(z)

        # 마지막 층은 sigmoid 사용 (이진 분류)
        if i == len(self.weights) - 1:
            a = sigmoid(z)
        else:
            a = self.activate(z)

        self.activations.append(a)

    return self.activations[-1]

# 클래스에 메서드 추가
NeuralNetwork.forward = forward

print("✅ 순전파 메서드 추가 완료!")

In [ ]:
# ============================================
# 6. 역전파 메서드 추가
# ============================================

def backward(self, X, y):
    m = X.shape[0]
    deltas = [None] * len(self.weights)

    # 출력층 오차
    output_error = self.activations[-1] - y
    deltas[-1] = output_error * sigmoid_derivative(self.activations[-1])

    # 역전파
    for i in range(len(self.weights) - 2, -1, -1):
        error = deltas[i+1] @ self.weights[i+1].T
        deltas[i] = error * self.activate_derivative(self.activations[i+1])

    # 가중치 업데이트
    gradient_norm = 0
    for i in range(len(self.weights)):
        dW = self.activations[i].T @ deltas[i] / m
        db = np.sum(deltas[i], axis=0, keepdims=True) / m

        gradient_norm += np.linalg.norm(dW)

        self.weights[i] -= self.learning_rate * dW
        self.biases[i] -= self.learning_rate * db

    return gradient_norm

# 클래스에 메서드 추가
NeuralNetwork.backward = backward

print("✅ 역전파 메서드 추가 완료!")

In [ ]:
# ============================================
# 7. 훈련 및 예측 메서드 추가
# ============================================

def train(self, X, y, epochs=5000, verbose=False):
    for epoch in range(epochs):
        # 순전파
        predictions = self.forward(X)

        # 손실 계산
        loss = np.mean((y - predictions) ** 2)
        self.loss_history.append(loss)

        # 역전파
        grad_norm = self.backward(X, y)
        self.gradient_norms.append(grad_norm)

        if verbose and (epoch + 1) % 1000 == 0:
            accuracy = np.mean((predictions > 0.5) == y) * 100
            print(f"Epoch {epoch+1:5d} | Loss: {loss:.6f} | Accuracy: {accuracy:.1f}%")

    return self.loss_history

def predict(self, X):
    output = self.forward(X)
    return (output > 0.5).astype(int)

# 클래스에 메서드 추가
NeuralNetwork.train = train
NeuralNetwork.predict = predict

print("✅ 훈련 및 예측 메서드 추가 완료!")

# 실습 1: 활성화 함수 비교 (Sigmoid vs ReLU vs Tanh)

In [ ]:
# ============================================
# 실습 1: 활성화 함수 비교 - 모델 훈련
# ============================================

print("=" * 60)
print("실습 1: 활성화 함수 비교 (Sigmoid vs ReLU vs Tanh)")
print("=" * 60)

activations = ['sigmoid', 'relu', 'tanh']
results = {}

for act in activations:
    print(f"\n🔄 {act.upper()} 학습 중...")
    nn = NeuralNetwork([2, 4, 1], activation=act, learning_rate=0.5)
    nn.train(X, y, epochs=5000)

    predictions = nn.predict(X)
    accuracy = np.mean(predictions == y) * 100

    results[act] = {
        'model': nn,
        'accuracy': accuracy,
        'final_loss': nn.loss_history[-1]
    }

    print(f"✅ 정확도: {accuracy:.1f}%")
    print(f"   최종 손실: {nn.loss_history[-1]:.6f}")

print("\n✅ 실습 1 훈련 완료!")

In [ ]:
# ============================================
# 실습 1: 활성화 함수 비교 - 결과 시각화
# ============================================

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 손실 비교
ax = axes[0, 0]
for act in activations:
    ax.plot(results[act]['model'].loss_history, label=act.upper(), linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss Comparison: Activation Functions')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 2. 정확도 비교
ax = axes[0, 1]
accs = [results[act]['accuracy'] for act in activations]
colors = ['blue', 'green', 'orange']
bars = ax.bar(activations, accs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Final Accuracy Comparison')
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 3. 수렴 속도 비교 (처음 1000 epoch)
ax = axes[1, 0]
for act in activations:
    ax.plot(results[act]['model'].loss_history[:1000], label=act.upper(), linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Convergence Speed (First 1000 epochs)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. 예측 결과
ax = axes[1, 1]
ax.axis('off')
text = "Final Predictions:\n\n"
for act in activations:
    nn = results[act]['model']
    preds = nn.predict(X)
    text += f"{act.upper()}:\n"
    for i in range(len(X)):
        text += f"  {X[i]} → {preds[i][0]} (정답: {y[i][0]})\n"
    text += "\n"

ax.text(0.1, 0.5, text, fontsize=11, family='monospace', verticalalignment='center')

plt.tight_layout()
plt.savefig('experiment1_activation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 실습 1 시각화 완료!")

# 실습 2: 층 깊이 실험 (1층, 2층, 5층)

In [ ]:
# ============================================
# 실습 2: 층 깊이 실험 - 모델 훈련
# ============================================

print("=" * 60)
print("실습 2: 층 깊이 실험 (1층, 2층, 5층)")
print("=" * 60)

layer_configs = [
    ([2, 4, 1], "1 Hidden Layer"),
    ([2, 4, 4, 1], "2 Hidden Layers"),
    ([2, 8, 6, 4, 3, 2, 1], "5 Hidden Layers")
]

depth_results = {}

for config, name in layer_configs:
    print(f"\n🔄 {name} 학습 중...")
    nn = NeuralNetwork(config, activation='relu', learning_rate=0.5)
    nn.train(X, y, epochs=5000)

    predictions = nn.predict(X)
    accuracy = np.mean(predictions == y) * 100

    depth_results[name] = {
        'model': nn,
        'accuracy': accuracy,
        'layers': len(config) - 1
    }

    print(f"✅ 정확도: {accuracy:.1f}%")

print("\n✅ 실습 2 훈련 완료!")

In [ ]:
# ============================================
# 실습 2: 층 깊이 실험 - 결과 시각화
# ============================================

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 손실 곡선
ax = axes[0, 0]
for name in depth_results.keys():
    ax.plot(depth_results[name]['model'].loss_history, label=name, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss vs Network Depth')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 2. 정확도
ax = axes[0, 1]
names = list(depth_results.keys())
accs = [depth_results[name]['accuracy'] for name in names]
colors = ['skyblue', 'lightgreen', 'salmon']
bars = ax.bar(names, accs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy vs Network Depth')
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 3. 기울기 변화
ax = axes[1, 0]
for name in depth_results.keys():
    gradients = depth_results[name]['model'].gradient_norms
    ax.plot(gradients, label=name, linewidth=2, alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Magnitude Over Training')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 4. 수렴 시간 분석
ax = axes[1, 1]
convergence_epochs = []
for name in depth_results.keys():
    loss_history = depth_results[name]['model'].loss_history
    # 손실이 0.01 이하로 떨어지는 시점
    conv_epoch = next((i for i, loss in enumerate(loss_history) if loss < 0.01), len(loss_history))
    convergence_epochs.append(conv_epoch)

bars = ax.bar(names, convergence_epochs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Epochs to Converge')
ax.set_title('Convergence Time (Loss < 0.01)')
ax.grid(True, alpha=0.3, axis='y')

for bar, epochs in zip(bars, convergence_epochs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{epochs}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('experiment2_depth.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 실습 2 시각화 완료!")

# 실습 3: 학습률 조정 (0.01, 0.1, 1.0)

In [ ]:
# ============================================
# 실습 3: 학습률 조정 - 모델 훈련
# ============================================

print("=" * 60)
print("실습 3: 학습률 조정 (0.01, 0.1, 1.0)")
print("=" * 60)

learning_rates = [0.01, 0.1, 1.0]
lr_results = {}

for lr in learning_rates:
    print(f"\n🔄 학습률 {lr} 학습 중...")
    nn = NeuralNetwork([2, 4, 1], activation='relu', learning_rate=lr)
    nn.train(X, y, epochs=5000)

    predictions = nn.predict(X)
    accuracy = np.mean(predictions == y) * 100

    lr_results[lr] = {
        'model': nn,
        'accuracy': accuracy
    }

    print(f"✅ 정확도: {accuracy:.1f}%")

print("\n✅ 실습 3 훈련 완료!")

In [ ]:
# ============================================
# 실습 3: 학습률 조정 - 결과 시각화
# ============================================

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 전체 손실 곡선
ax = axes[0, 0]
for lr in learning_rates:
    ax.plot(lr_results[lr]['model'].loss_history, label=f'LR={lr}', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss vs Learning Rate')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 2. 초기 500 epochs (수렴 속도)
ax = axes[0, 1]
for lr in learning_rates:
    ax.plot(lr_results[lr]['model'].loss_history[:500], label=f'LR={lr}', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Early Training (First 500 epochs)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. 최종 정확도
ax = axes[1, 0]
lrs_str = [str(lr) for lr in learning_rates]
accs = [lr_results[lr]['accuracy'] for lr in learning_rates]
colors = ['lightcoral', 'lightblue', 'lightgreen']
bars = ax.bar(lrs_str, accs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Final Accuracy vs Learning Rate')
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 4. 학습 안정성 (손실 변동성)
ax = axes[1, 1]
volatilities = []
for lr in learning_rates:
    loss_history = lr_results[lr]['model'].loss_history
    # 마지막 1000 epochs의 표준편차
    volatility = np.std(loss_history[-1000:])
    volatilities.append(volatility)

bars = ax.bar(lrs_str, volatilities, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Loss Volatility (Std Dev)')
ax.set_title('Training Stability (Last 1000 epochs)')
ax.grid(True, alpha=0.3, axis='y')

for bar, vol in zip(bars, volatilities):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{vol:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('experiment3_learning_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 실습 3 시각화 완료!")

# 실습 4: 기울기 확인 (각 층의 기울기 크기 분석)

In [ ]:
# ============================================
# 실습 4: 기울기 확인 - 깊은 신경망 훈련
# ============================================

print("=" * 60)
print("실습 4: 각 층의 기울기 크기 확인")
print("=" * 60)

# 깊은 신경망으로 기울기 추적
print("\n🔄 깊은 신경망 (5층) 학습 중...")
nn_deep = NeuralNetwork([2, 8, 6, 4, 3, 1], activation='relu', learning_rate=0.5)

# 기울기 추적을 위한 수정된 학습
layer_gradients = {i: [] for i in range(len(nn_deep.weights))}

for epoch in range(2000):
    # 순전파
    predictions = nn_deep.forward(X)

    # 역전파 및 기울기 저장
    m = X.shape[0]
    deltas = [None] * len(nn_deep.weights)

    output_error = nn_deep.activations[-1] - y
    deltas[-1] = output_error * sigmoid_derivative(nn_deep.activations[-1])

    for i in range(len(nn_deep.weights) - 2, -1, -1):
        error = deltas[i+1] @ nn_deep.weights[i+1].T
        deltas[i] = error * nn_deep.activate_derivative(nn_deep.activations[i+1])

    # 각 층의 기울기 크기 저장
    for i in range(len(nn_deep.weights)):
        dW = nn_deep.activations[i].T @ deltas[i] / m
        grad_norm = np.linalg.norm(dW)
        layer_gradients[i].append(grad_norm)

        nn_deep.weights[i] -= nn_deep.learning_rate * dW
        nn_deep.biases[i] -= nn_deep.learning_rate * np.sum(deltas[i], axis=0, keepdims=True) / m

    if (epoch + 1) % 500 == 0:
        loss = np.mean((y - predictions) ** 2)
        print(f"Epoch {epoch+1}: Loss = {loss:.6f}")

print("✅ 학습 완료!")

In [ ]:
# ============================================
# 실습 4: 기울기 확인 - 결과 시각화
# ============================================

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 모든 층의 기울기 변화
ax = axes[0, 0]
for i in range(len(nn_deep.weights)):
    ax.plot(layer_gradients[i], label=f'Layer {i+1}', linewidth=2, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Magnitude in Each Layer')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 2. 초기 기울기 (처음 200 epochs)
ax = axes[0, 1]
for i in range(len(nn_deep.weights)):
    ax.plot(layer_gradients[i][:200], label=f'Layer {i+1}', linewidth=2, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Gradient Norm')
ax.set_title('Early Gradients (First 200 epochs)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. 평균 기울기 크기 (층별)
ax = axes[1, 0]
avg_gradients = [np.mean(layer_gradients[i]) for i in range(len(nn_deep.weights))]
layer_names = [f'Layer {i+1}' for i in range(len(nn_deep.weights))]
colors_grad = plt.cm.viridis(np.linspace(0, 1, len(nn_deep.weights)))

bars = ax.bar(layer_names, avg_gradients, color=colors_grad, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Average Gradient Norm')
ax.set_title('Average Gradient Magnitude per Layer')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

for bar, grad in zip(bars, avg_gradients):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{grad:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. 기울기 비율 (층 간 비교)
ax = axes[1, 1]
gradient_ratios = []
for i in range(len(nn_deep.weights) - 1):
    ratio = np.mean(layer_gradients[i]) / np.mean(layer_gradients[i+1])
    gradient_ratios.append(ratio)

ratio_labels = [f'L{i+1}/L{i+2}' for i in range(len(gradient_ratios))]
bars = ax.bar(ratio_labels, gradient_ratios, color=colors_grad[:-1], alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Gradient Ratio')
ax.set_title('Gradient Flow Between Layers')
ax.axhline(y=1.0, color='r', linestyle='--', linewidth=2, label='Equal Gradient')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

for bar, ratio in zip(bars, gradient_ratios):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{ratio:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('experiment4_gradients.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 실습 4 시각화 완료!")

# 종합 결과 요약 및 최종 비교

In [ ]:
# ============================================
# 종합 결과 요약
# ============================================

print("=" * 60)
print("📊 전체 실습 결과 요약")
print("=" * 60)

print("\n1️⃣ 활성화 함수 비교:")
for act in activations:
    print(f"   {act.upper():8s}: 정확도 {results[act]['accuracy']:.1f}%, "
          f"최종 손실 {results[act]['final_loss']:.6f}")

print("\n2️⃣ 층 깊이 실험:")
for name in depth_results.keys():
    print(f"   {name:20s}: 정확도 {depth_results[name]['accuracy']:.1f}%")

print("\n3️⃣ 학습률 조정:")
for lr in learning_rates:
    print(f"   LR = {lr:4.2f}: 정확도 {lr_results[lr]['accuracy']:.1f}%")

print("\n4️⃣ 기울기 분석:")
print(f"   총 {len(nn_deep.weights)}개 층의 기울기 추적 완료")
print(f"   평균 기울기 범위: {min(avg_gradients):.4f} ~ {max(avg_gradients):.4f}")

print("\n" + "=" * 60)
print("🎉 모든 실습 완료! 결과 이미지가 저장되었습니다.")
print("=" * 60)

In [ ]:
# ============================================
# 최종 비교 그래프 생성
# ============================================

# 최종 비교 그래프
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Neural Network Experiments - Summary', fontsize=16, fontweight='bold')

# 1. 활성화 함수별 최종 손실
ax = axes[0, 0]
final_losses = [results[act]['final_loss'] for act in activations]
colors1 = ['blue', 'green', 'orange']
ax.bar(activations, final_losses, color=colors1, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Final Loss')
ax.set_title('Experiment 1: Activation Functions')
ax.grid(True, alpha=0.3, axis='y')

# 2. 네트워크 깊이별 정확도
ax = axes[0, 1]
names = list(depth_results.keys())
accs_depth = [depth_results[name]['accuracy'] for name in names]
colors2 = ['skyblue', 'lightgreen', 'salmon']
ax.bar(range(len(names)), accs_depth, color=colors2, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(['1 Layer', '2 Layers', '5 Layers'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Experiment 2: Network Depth')
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.3, axis='y')

# 3. 학습률별 수렴 속도
ax = axes[1, 0]
for lr in learning_rates:
    loss_hist = lr_results[lr]['model'].loss_history
    ax.plot(loss_hist[:1000], label=f'LR={lr}', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Experiment 3: Learning Rate')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 4. 층별 평균 기울기
ax = axes[1, 1]
layer_labels = [f'L{i+1}' for i in range(len(avg_gradients))]
colors4 = plt.cm.viridis(np.linspace(0, 1, len(avg_gradients)))
ax.bar(layer_labels, avg_gradients, color=colors4, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Average Gradient Norm')
ax.set_title('Experiment 4: Gradient Analysis')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('experiment_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 요약 그래프 생성 완료!")
print("\n💡 Tip: 각 파라미터를 수정하여 다시 실행해보세요!")